# wildfire_grpo_train_hf — GRPO only (Hugging Face GPU)


Use this notebook only for GRPO training on the Hugging Face GPU runtime.

Cell order:
1. Run Cell 1 first each session (loads env vars + BASE).
2. Run Cell 2 to clone/update and install dependencies.
3. Run Cell 3 for OpenEnv/training preflight.
4. Run Cell 4 for the GRPO training run.

`grpo_wildfire/log.jsonl` is written by `train_grpo.py` (often not in git; large artifacts stay local).

## Training configuration actually used for the submission

The hackathon submission was trained with the **`deadline_v2_a10g`**
configuration in Cell 4 below, which is scheduled for **20 GRPO iterations**:

- `total_iterations=20`
- `task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20))`
- `group_size=4`, `seeds_per_iter=2`, `micro_batch_size=2` — chosen to fill
  the A10G (≈22 GB VRAM) without OOM
- `lora_dropout=0.0`, `warmup_iters=3`, `save_every=6`
- LoRA shape: rank 16, alpha 32, targets `q/k/v/o_proj` (defaults from
  `train_grpo.py`)

**We stopped after iteration 17** due to the hackathon compute budget.
`train_grpo.py` is resume-safe: each iter writes
`grpo_wildfire/latest/`, `optimizer_state.pt`, and `resume_state.json`,
so the iter-17 checkpoint is the one used for the trained-adapter
evaluation in `submission_artifacts/eval_trained.json` (promoted to
`grpo_wildfire/final_adapter/` before eval).

After training finishes, use `notebooks/wildfire_eval_plots_hf.ipynb` for
the OpenEnv baseline/trained evaluation and final artifacts.

In [ ]:
# Cell 1 (run first each session): env vars + constants (HF runtime)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens available: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "matplotlib", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl",
    ],
    check=True,
)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")

In [ ]:
# Cell 4: GRPO training run (resume-safe).
#
# This is the exact `deadline_v2_a10g` configuration used for the hackathon
# submission: 20 GRPO iterations on a single A10G with bigger group/microbatch
# knobs to fill the 23 GB VRAM. We actually stopped after iteration 17 due
# to the compute budget; the resumed iter-17 adapter is what
# `submission_artifacts/eval_trained.json` was generated from.
#
# `train_grpo.py` is resume-safe: every iter writes
# `grpo_wildfire/latest/`, `optimizer_state.pt`, and `resume_state.json`,
# so rerunning this cell loads the latest LoRA + optimizer state and picks
# up from `next_iter`.

import subprocess, sys, os

_HERE = os.path.dirname(os.path.abspath(globals().get("__file__", ".")))
_PROJECT_ROOT = os.path.abspath(os.path.join(_HERE, ".."))
if not os.path.isfile(os.path.join(_PROJECT_ROOT, "train_grpo.py")):
    _PROJECT_ROOT = os.path.expanduser("~/app/wildfire-env")

subprocess.run([
    sys.executable, "-c",
    f"""
import sys
sys.path.insert(0, {_PROJECT_ROOT!r})
from train_grpo import Config, train

train(Config(
    total_iterations=20,
    task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20)),
    group_size=4,           # 2x rollout parallelism per iter
    seeds_per_iter=2,       # more env variance per iter
    micro_batch_size=2,     # fills the 23 GB A10G better
    max_episode_steps=20,
    lora_dropout=0.0,
    warmup_iters=3,
    save_every=6,
    run_name="deadline_v2_a10g",
))
"""
], cwd=_PROJECT_ROOT, check=True)

print("deadline_v2_a10g done (or stopped after iter 17 in our run). "
      "See grpo_wildfire/log.jsonl, grpo_wildfire/latest (resume), "
      "grpo_wildfire/final_adapter (eval).")


# Stop here

After training, use `notebooks/wildfire_eval_plots_hf.ipynb` for OpenEnv eval, training-curve plots, and submission artifacts.
